# 03 · NoLA experiments

`load_data` → `load_clip` → `make_cde` → `score_clip` → **`select_anchors` and `extract_dino_features`** → `train_dino_head` → `fine_tune_clip` → `evaluate_full_run`.

Baseline anchor selection needs only CLIP scores. Frozen DINO extraction needs only the images, not the anchors. Either can go first in the baseline; the DINO-assisted experiment extracts features first and feeds them into its modified anchor selection. The DINO head needs both. Run only the sections you need; the CDE experiment stops before either branch. All research comparisons use validation, not test.

The notebook collects CSVs only. Interpretation is added as Markdown after reviewing the saved predictions and images offline.

Use the companion four-dataset batch notebook to run this same design on EuroSAT, Oxford Pets, CIFAR-10, and DTD.


## Setup


In [ ]:
from pathlib import Path
import hashlib
import inspect
import json
import subprocess
import sys

COLAB = True
DATASETS = ('eurosat', 'oxford_pets', 'cifar10', 'dtd')
TAG = DATASETS[0]  # Change this for an interactive single-dataset run.
RUN_NAME = 'four-dataset-ablations-v1'
SEED = 7777
HEAD_EPOCHS, PROMPT_EPOCHS = 50, 50
BATCH, MICRO, EXTRACT_BATCH = 512, 32, 32

if COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/NoLA-Final-Evaluation')
    ROOT = Path('/content/nola')
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'gdown', 'ftfy', 'wget', 'timm>=1.0.22', 'transformers>=4.53', 'sentencepiece'], check=True)
else:
    ROOT = Path.home() / 'nola-walkthrough'
    BASE = ROOT / 'final-evaluation'
    sys.path.insert(0, str(ROOT / '.deps'))

sys.path.insert(0, str(BASE))
from setup_inputs import setup

ROOT, DATA = setup(ROOT, ROOT/'data', [TAG], source=BASE/'source.zip')

import torch
from torch.nn.functional import normalize
from IPython.display import Code, display
import nola_eval
import experiments as ex


## Baseline stages

These named calls are the pipeline. `nola_eval.py` supplies the standard implementation; the functions below make each stage’s inputs and outputs explicit. The experimental sections change only the inputs or the stage named in the question.


In [ ]:
def load_data():
    nola_eval.seed_all(SEED)
    splits, source, names, bank_name = nola_eval.load_data(TAG, DATA, seed=SEED)
    rows = {part: splits[part] for part in ('train', 'val')}
    truth = {part: torch.tensor([y for _, y, _ in group]) for part, group in rows.items()}
    return rows, source, names, bank_name, truth


def load_clip():
    weights = torch.jit.load(str(ROOT / 'weights/ViT-B-32.pt'), map_location='cpu')
    model = nola_eval.clip.build_model(weights.state_dict())
    return model.float().cuda().eval().requires_grad_(False)


def make_cde(clip, names, bank_name):
    descriptions, templates = nola_eval.get_prompts(ROOT, bank_name)
    prompt_parts = [(descriptions[name], [t.format(name) for t in templates]) for name in names]
    prompts = [desc + tmpl for desc, tmpl in prompt_parts]
    return nola_eval.build_cde_from_prompts(clip, prompts), prompt_parts, prompts


def encode_clip_images(clip, rows, source):
    return nola_eval.extract_features(nola_eval.clip_scores, clip, rows, source)


def score_clip(features, cde, scale=1.0):
    return nola_eval.class_scores(features, cde, scale)


def select_anchors(train_logits, cap):
    return nola_eval.select_anchors(train_logits, cap)


def extract_dino_features(rows, source):
    model = nola_eval.vit_base(patch_size=16, num_classes=0)
    weights = ROOT / 'weights/dino_vitbase16_pretrain.pth'
    model.load_state_dict(torch.load(weights, map_location='cpu', weights_only=True))
    model.cuda().eval().requires_grad_(False)
    return nola_eval.extract_features(nola_eval.dino_features, model, rows, source)


def train_dino_head(features, anchors, classes):
    head = nola_eval.train_alignment_stage(features, anchors, classes, HEAD_EPOCHS, SEED)
    return nola_eval.alignment_predictions(head, features)


def fine_tune_clip(clip, cde, rows, source, pseudo_labels, truth, folder):
    folder.mkdir(parents=True, exist_ok=True)
    return nola_eval.train_visual_prompt_stage(
        clip, cde, rows, source, pseudo_labels, truth, folder,
        PROMPT_EPOCHS, BATCH, MICRO, SEED,
    )


def evaluate_full_run(folder, arm, rows, source, truth, names, logits, aligned,
                      tuned, anchors, epoch, curve):
    nola_eval.export_baseline(
        folder, dict(**key, arm=arm), rows, source, truth, names, logits, aligned,
        tuned, *anchors, epoch, curve, HEAD_EPOCHS, MICRO,
    )


## Inspect the implementation in Colab


In [ ]:
# Display any stage above, or its underlying implementation.
display(Code(inspect.getsource(nola_eval.fit_alignment), language='python'))


## Shared prefix: load data and models, make the baseline CDE


In [ ]:
rows, source, names, bank_name, truth = load_data()
out = BASE/'runs'/'03-experiments'/RUN_NAME/TAG
out.mkdir(parents=True, exist_ok=True)
split_json = json.dumps(rows)
split_id = hashlib.sha256(split_json.encode()).hexdigest()
key = dict(run_id=f'{RUN_NAME}/{TAG}', dataset=TAG, seed=SEED, split_id=split_id)
(out/'split.json').write_text(split_json)
(out/'manifest.json').write_text(json.dumps(dict(**key, head_epochs=HEAD_EPOCHS,
    prompt_epochs=PROMPT_EPOCHS, batch=BATCH, micro=MICRO, stage='development_validation',
    sources={name: nola_eval.sha(BASE/name) for name in ('nola_eval.py', 'experiments.py')}), indent=2))

clip = load_clip()
release_cde, prompt_parts, release_prompts = make_cde(clip, names, bank_name)
clip_scale = clip.logit_scale.detach().exp().item()
clip_val = encode_clip_images(clip, {'val': rows['val']}, source)
release_val = score_clip(clip_val, release_cde)


## Experiment 1 · CDE only

**Question.** Do different ways of combining templates and descriptions improve frozen CLIP’s zero-shot predictions?

**Test.** Keep CLIP, validation rows and image embeddings fixed. Change only the prompt bank or score reduction. Compare each change with the relevant control: template addition against descriptions only, merges against unmerged prompts, and variation blends against the suffix mean. This section does not select anchors, load DINO, or fine-tune CLIP.


In [ ]:
text_banks = {}
for descriptions, templates in prompt_parts:
    d = nola_eval.encode_texts(clip, descriptions)
    t = nola_eval.encode_texts(clip, templates)
    # Historical suffix: each formatted template follows each description, before tokenization.
    suffix = [f'{p.rstrip()} {q}' for p in descriptions for q in templates]
    # Literal inverse order: append each description to each formatted template.
    template_suffix = [f'{q.rstrip()} {p}' for p in descriptions for q in templates]
    sx = nola_eval.encode_texts(clip, suffix)
    tx = nola_eval.encode_texts(clip, template_suffix)
    # Group near-duplicate prompts before merging or blending.
    sx_by_desc = normalize(sx.reshape(len(d), len(t), -1).mean(1), dim=1)
    mean = ex.centroid(sx_by_desc)
    groups = ex.clusters(sx_by_desc, .75)
    directions = torch.stack([ex.centroid(sx_by_desc[g]) for g in groups])
    weights = torch.tensor([len(g) for g in groups], dtype=mean.dtype)
    banks = dict(descriptions=ex.centroid(d)[None],
                 template_add=normalize(ex.centroid(d) + ex.centroid(t), dim=0)[None],
                 suffix=ex.centroid(sx)[None], template_suffix=ex.centroid(tx)[None],
                 unmerged=d, suffix_mean=mean[None])
    for cutoff, name in [(.75, 'merge075'), (.80, 'merge080')]:
        banks[name] = torch.stack([ex.centroid(d[g]) for g in ex.clusters(d, cutoff)])
    for gate, name in [(.75, 'blend075'), (.90, 'blend090')]:
        keep = directions @ mean < gate
        v = normalize(len(d) * mean + weights[keep] @ directions[keep], dim=0) if keep.any() else mean
        banks[name] = v[None]
    for kind, prototypes in banks.items():
        text_banks.setdefault(kind, []).append(prototypes)


In [ ]:
text_scores = {'release': release_val['val']}
for arm, prototypes in text_banks.items():
    text_scores[arm] = ex.score(clip_val['val'], prototypes)
text_predictions = {arm: scores.argmax(1) for arm, scores in text_scores.items()}
text_contrasts = [
    ('descriptions', 'release'), ('template_add', 'descriptions'),
    ('suffix', 'release'), ('template_suffix', 'release'), ('template_suffix', 'suffix'),
    ('unmerged', 'descriptions'), ('merge075', 'unmerged'), ('merge080', 'unmerged'),
    ('blend075', 'suffix_mean'), ('blend090', 'suffix_mean'),
]


In [ ]:
# Save the scores as per-image predictions for paired analysis offline.
val_ids = [row[0] for row in rows['val']]
records = [dict(**key, arm=arm, split='val', stage='cde', image_id=image_id,
                truth=y, prediction=label)
           for arm, pred in text_predictions.items()
           for image_id, y, label in zip(val_ids, truth['val'].tolist(), pred.tolist())]
nola_eval.write_csv(out / 'cde_predictions.csv', records)

metrics = []
for arm, pred in text_predictions.items():
    correct, count = nola_eval.top1_counts(pred, truth['val'])
    metrics.append(dict(**key, arm=arm, split='val', stage='cde', correct=correct,
                        n=count, top1_pct=100 * correct / count))
nola_eval.write_csv(out / 'cde_validation.csv', metrics)
nola_eval.write_csv(out / 'cde_contrasts.csv',
                    [dict(**key, arm=arm, control=control) for arm, control in text_contrasts])


### Interpretation · CDE only

Pending offline review of `cde_validation.csv` and `cde_predictions.csv`: report paired accuracy changes against the declared controls, then describe representative class confusions and validation tiles. No downstream claim follows from this section.


## Full-run prefix: score the training pool


In [ ]:
clip_train = encode_clip_images(clip, {'train': rows['train']}, source)
clip_features = {**clip_train, **clip_val}
release_cosine = score_clip(clip_features, release_cde)
release_logits = score_clip(clip_features, release_cde, clip_scale)
del clip
release_cap = max(16, int(.2 * len(rows['train']) / len(names)))


## Full-run prefix: extract frozen DINO features

Run this cell before the full experiments. It does not depend on the anchor IDs or on completing baseline training.

In [ ]:
dino_features = extract_dino_features(rows, source)


## Native baseline · reuse Notebook 02 exports

The unchanged NoLA reference for each dataset is in `runs/baseline-full-v4/<dataset>`. Do not train it again here. The matched ablation controls below change the selection rule and are not interchangeable with the native baseline.


### Interpretation · baseline

Pending offline review of the stage metrics, anchor counts and purity, and representative validation mistakes. The final result will refer to the selected validation checkpoint, not necessarily the last epoch.


## Experiment 3 · DINO-assisted anchor selection

**Question.** Does frozen DINO similarity help CLIP choose better anchors without adding more anchors than the matched control?

**Test.** Extract DINO features before this modified selector. Compare DINO neighbor agreement with and without CLIP-posterior reweighting. Separately compare confidence-only expansion with expansion that adds a DINO-neighbor score. Within each pair keep candidate ownership, pseudo-labels, and per-class anchor counts matched. Then run the unchanged alignment and CLIP tuning stages.


In [ ]:
cap = max(1, len(rows['train']) // (5*len(names)))
seed_pools = ex.candidates(release_logits['train'], cap)
anchor_arms = {}
# Rank CLIP nominees by agreement among frozen DINO neighbors.
x = normalize(dino_features['train'].float().cuda(), dim=1)
p = release_logits['train'].softmax(1).cuda()
label = p.argmax(1)
k, block = min(20, len(x)-1), 256
neighbors, distances = [], []
for start in range(0, len(x), block):
    cosine = x[start:start + block] @ x.T
    r = torch.arange(len(cosine), device=x.device)
    cosine[r, start + r] = -torch.inf
    vals, idx = cosine.topk(k, dim=1)
    neighbors.append(idx)
    distances.append((1. - vals).clamp_min(1e-6))
idx, dist = torch.cat(neighbors), torch.cat(distances)
scale = dist[:, -1]
weight = torch.exp(-dist.square() / (scale[:, None] * scale[idx]))
same = (label[idx] == label[:, None]).float()
overlap = (p.sqrt()[:, None, :] * p.sqrt()[idx]).sum(2).clamp(0, 1)
# Change only the graph edge weights.
modified = weight * (.25 + .75 * overlap)
agreement = [((w * same).sum(1) / w.sum(1).clamp_min(1e-12)).cpu() for w in (weight, modified)]
ranks = []
for values in agreement:
    class_pools = [torch.where(label.cpu() == c)[0] for c in range(p.shape[1])]
    ranks.append([r[values[r].argsort(descending=True, stable=True)] for r in class_pools])
first = idx[0, :min(5, k)]
graph_trace = dict(neighbors=first.cpu().tolist(), dino_weights=weight[0, :len(first)].cpu().tolist(),
             reweighted=modified[0, :len(first)].cpu().tolist())
rankings = ranks
quota = [len(p) for p in seed_pools]
for mode, rank in zip(('dino_only', 'dino_clip'), rankings):
    anchor_arms[mode] = ex.pack(rank, quota)
print('One train image and its DINO neighbors:', rows['train'][0][0],
      [rows['train'][i][0] for i in graph_trace['neighbors']])
print('Same edges: original → reweighted', graph_trace['dino_weights'], graph_trace['reweighted'])


In [ ]:
# Keep the CLIP seed pool fixed; change the ranking for extra anchors.
cosine = release_cosine['train']
features = dino_features['train']
seeds = seed_pools
x = normalize(features, dim=1)
used = torch.zeros(len(x), dtype=torch.bool)
used[torch.cat(seeds)] = True
under = [c for c, ids in enumerate(seeds) if 0 < len(ids) < cap]
clip_only = clip_dino = None
if under:
    under = torch.tensor(under)
    owner = under[cosine[:, under].argmax(1)]
    gap = cosine.max(1).values - cosine[torch.arange(len(x)), owner]
    log_p = (clip_scale*cosine).log_softmax(1)
    baseline, retrieved, added = [], [], []
    for c, ids in enumerate(seeds):
        pool = torch.where((owner == c) & (~used) & (gap <= .06))[0]
        budget = min(cap-len(ids), min(64, max(8, cap//2)), len(pool))
        if budget:
            confidence = log_p[pool, c]
            nearest = (x[pool] @ x[ids].T).topk(min(5, len(ids)), dim=1).values.mean(1)
            base = pool[confidence.argsort(descending=True, stable=True)[:budget]]
            new = pool[(confidence + .5*nearest).argsort(descending=True, stable=True)[:budget]]
        else:
            base = new = pool[:0]
        added.append(budget)
        baseline.append(torch.cat((ids, base)))
        retrieved.append(torch.cat((ids, new)))
    if sum(added):
        quota = [len(ids) for ids in baseline]
        clip_only = ex.pack(baseline, quota)
        clip_dino = ex.pack(retrieved, quota)
if clip_only is not None:
    anchor_arms.update(clip_expansion=clip_only, dino_retrieval=clip_dino)
else:
    print('Seed-retrieval comparison skipped: no eligible additions.')


In [ ]:
for arm, anchors in anchor_arms.items():
    folder = out / 'anchors' / arm
    aligned = train_dino_head(dino_features, anchors, len(names))
    if not (folder / 'complete.json').exists():
        model, epoch, curve = fine_tune_clip(
            load_clip(), release_cde, rows, source, aligned['train'], truth, folder,
        )
        evaluate_full_run(folder, arm, rows, source, truth, names,
                          release_logits, aligned, model, anchors, epoch, curve)
        del model
        (folder / 'complete.json').write_text(json.dumps(dict(epoch=epoch)))


### Interpretation · anchor selection

Pending offline review of per-class anchor counts, purity, and paired alignment and final validation results. Compare reweighted DINO agreement against unweighted agreement, and DINO-assisted expansion against confidence-only expansion when eligible. Describe representative anchors and validation mistakes after checking the saved rows.


## Experiment 2 · substitute frozen encoders

**Question.** Does SigLIP 2 improve anchor nomination, does DINOv3 improve the alignment features, and do either of those changes help the final classifier?

**Test.** Cross CLIP/SigLIP 2 nomination with DINO/DINOv3 features. Match top-1 anchor counts per class across the two nomination encoders. Change one factor in each paired comparison. Every arm uses the same final CLIP visual-prompt classifier, initialized from the original CLIP CDE. The matched CLIP+DINO arm, not the native baseline, is the control.


In [ ]:
siglip = ROOT/'weights/siglip2-base-patch16-224'
dinov3 = ROOT/'weights/convnext_tiny.dinov3_lvd1689m.safetensors'
# Download only when this experiment is selected; verify pinned files below.
from huggingface_hub import snapshot_download, hf_hub_download
import shutil
if not (siglip / 'model.safetensors').exists():
    snapshot_download('google/siglip2-base-patch16-224', revision=ex.SIGLIP_REV,
                      allow_patterns=['*.json', '*.model', 'model.safetensors'], local_dir=siglip)
if not dinov3.exists():
    weight = hf_hub_download('timm/' + ex.V3_MODEL, 'model.safetensors', revision=ex.V3_REV)
    shutil.copyfile(weight, dinov3)
issues = [ex.verify(siglip/'model.safetensors', ex.SIGLIP_SHA), ex.verify(dinov3, ex.V3_SHA)]
if any(issues):
    raise FileNotFoundError('; '.join(problem for problem in issues if problem))

siglip_images, siglip_bank = ex.siglip_embeddings(
    siglip, rows, source, release_prompts, nola_eval.loader, EXTRACT_BATCH)
siglip_logits = score_clip(siglip_images, siglip_bank, clip_scale)
v3_features, v3_preprocessing = ex.dinov3_features(dinov3, rows, source, nola_eval.loader, EXTRACT_BATCH)

# Match anchor counts across nomination encoders before crossing with DINO features.
nomination = {'clip': release_logits, 'siglip2': siglip_logits}
encoder_cap = max(1, len(rows['train']) // (5*len(names)))
pools = {name: ex.candidates(scores['train'], encoder_cap)
         for name, scores in nomination.items()}
quota = [min(len(pools['clip'][c]), len(pools['siglip2'][c])) for c in range(len(names))]
matched_anchors = {name: ex.pack(pool, quota) for name, pool in pools.items()}


In [ ]:
for vlm, logits in nomination.items():
    for visual, features in [('dino', dino_features), ('dinov3', v3_features)]:
        arm = f'{vlm}_{visual}'
        folder = out / 'encoders' / arm
        anchors = matched_anchors[vlm]
        aligned = train_dino_head(features, anchors, len(names))
        if not (folder / 'complete.json').exists():
            model, epoch, curve = fine_tune_clip(
                load_clip(), release_cde, rows, source, aligned['train'], truth, folder,
            )
            evaluate_full_run(folder, arm, rows, source, truth, names,
                              logits, aligned, model, anchors, epoch, curve)
            del model
            (folder / 'complete.json').write_text(json.dumps(dict(epoch=epoch)))


### Interpretation · frozen encoders

Pending offline review of matched CLIP vs. SigLIP 2 at each feature encoder, and DINO vs. DINOv3 at each nomination encoder. Report paired accuracy changes, whether alignment gains survive prompt tuning, and examples of corrected and regressed tiles.


## Presentation artifacts

The notebook writes raw CSVs. Run `python research_figures.py <run-folder>` separately to produce paired ablation tables, learning curves, slide PNGs and the combined PDF. Missing arms are omitted, not plotted as zero.
